# 02 — Spark Architecture

**Module 2 of 15**

## Learning Objectives
- Describe the Driver / Executor model
- Explain the role of the Cluster Manager (Standalone, YARN, Kubernetes, local)
- Understand Jobs, Stages, and Tasks
- Understand partitions and why they matter
- Understand lazy evaluation and the DAG (Directed Acyclic Graph)
- Understand transformations vs actions

---


## 2.1 The Big Picture

Every Spark application, no matter how simple, follows the same architecture:

```
                     ┌──────────────────────────┐
                     │        Driver Program      │
                     │  (runs your main() code,   │
                     │   creates SparkSession,     │
                     │   builds the DAG, schedules │
                     │   tasks)                    │
                     └─────────────┬────────────┘
                                   │
                    talks to the Cluster Manager
                                   │
              ┌────────────────────┼────────────────────┐
              │                    │                    │
       ┌──────▼─────┐       ┌──────▼─────┐       ┌──────▼─────┐
       │  Executor 1  │       │  Executor 2  │       │  Executor 3  │
       │ (JVM process)│       │ (JVM process)│       │ (JVM process)│
       │  runs Tasks  │       │  runs Tasks  │       │  runs Tasks  │
       │  caches data │       │  caches data │       │  caches data │
       └──────────────┘       └──────────────┘       └──────────────┘
```

- **Driver**: the process running your Python/Scala code. It owns the `SparkSession`, converts your DataFrame/RDD operations into a logical plan, optimizes it, splits it into tasks, and sends those tasks to executors. It also collects results back (e.g., for `.collect()` or `.show()`).
- **Executors**: worker processes (JVMs) that actually run the tasks and store data partitions in memory or disk (for caching). Each executor has a fixed number of cores and a fixed amount of memory.
- **Cluster Manager**: allocates resources (CPU, memory) for the driver and executors across a cluster of machines. Options: Spark Standalone, Hadoop YARN, Kubernetes, Apache Mesos (deprecated), or simply `local[*]` for running everything on your own laptop.


## 2.2 Jobs, Stages, and Tasks

When you call an **action** (like `.count()` or `.collect()`), Spark builds an execution plan broken into three levels of granularity:

| Level | Definition |
|---|---|
| **Job** | Triggered by one action. E.g., calling `df.count()` launches one job. |
| **Stage** | A job is split into stages at points where data must be shuffled across the network (e.g., a `groupBy` or `join`). Tasks within a stage can run without waiting on network shuffles. |
| **Task** | The smallest unit of work — one task processes one partition of data on one executor core. |

**Rule of thumb:** more partitions = more parallelism, but too many small partitions adds scheduling overhead; too few large partitions under-utilizes your cluster. Balancing this is called partition tuning (covered in Notebook 11).


## 2.3 Partitions

Spark splits your data into **partitions** — chunks of data that are distributed across the cluster and processed in parallel.

- A DataFrame with 400 million rows might be split into 200 partitions of 2 million rows each.
- Each partition is processed independently by one task on one executor core.
- The number of partitions directly controls your degree of parallelism.

Think of partitions like dealing a deck of cards to multiple players — each player (executor core) works on their own hand (partition) simultaneously, instead of one person going through the entire deck alone.


## 2.4 Transformations vs Actions — Lazy Evaluation

This is one of the most important concepts in Spark.

- **Transformations** (`.filter()`, `.select()`, `.groupBy()`, `.withColumn()`, `.join()`, ...) are **lazy**. They don't execute immediately — Spark just records "here's what I need to do" and builds up a logical plan (a DAG — Directed Acyclic Graph of operations).
- **Actions** (`.show()`, `.collect()`, `.count()`, `.write.save()`, ...) **trigger execution**. Only when an action is called does Spark:
  1. Take the recorded logical plan,
  2. Optimize it (via the Catalyst optimizer, for DataFrames/SQL),
  3. Convert it into physical stages and tasks,
  4. Execute it across the cluster.

**Why is this useful?** Because Spark can look at your *entire* chain of transformations before running anything, and optimize the whole pipeline — e.g., pushing filters down before joins, skipping columns you never use, combining multiple steps into a single pass over the data. This is fundamentally different from pandas, where every line executes immediately.


In [ ]:
# Conceptual illustration (no Spark session needed yet — this is pseudocode-style Python
# to visualize what "lazy" means). We'll do this for real with Spark starting in Notebook 04.

class LazyPipeline:
    def __init__(self):
        self.steps = []

    def filter(self, condition_desc):
        print(f"[recorded] filter: {condition_desc}")
        self.steps.append(("filter", condition_desc))
        return self  # nothing executes yet

    def select(self, cols):
        print(f"[recorded] select: {cols}")
        self.steps.append(("select", cols))
        return self

    def show(self):
        print("\n--- ACTION CALLED: executing the full recorded plan now ---")
        for step, desc in self.steps:
            print(f"  running {step} -> {desc}")
        print("--- done ---")

pipeline = LazyPipeline()
pipeline.filter("age > 30").select(["name", "age"])
print("\nNotice: nothing has 'run' yet, we only recorded the plan.")
pipeline.show()  # only now does execution happen


## 2.5 Fault Tolerance

Spark achieves fault tolerance through **lineage**: because the DAG records every transformation from the original data source, if a partition is lost (e.g., an executor crashes), Spark can simply recompute just that partition by replaying the recorded transformations — no need to restart the whole job. This is a key advantage of the RDD/DAG model over simply storing intermediate results with no record of how they were derived.


## 2.6 Cluster Manager Options

| Cluster Manager | Typical Use Case |
|---|---|
| `local[*]` | Development/learning on your own laptop; uses all local CPU cores as "executors" |
| Spark Standalone | Spark's own simple built-in cluster manager |
| Hadoop YARN | Common in on-prem Hadoop ecosystems (Cloudera, Hortonworks) |
| Kubernetes | Modern cloud-native deployments; each executor is a pod |
| Databricks / EMR / Dataproc | Managed cloud platforms that handle cluster management for you |

Throughout this course, we will use `local[*]` mode so everything runs directly on your machine — the code you write is *identical* to what you'd run on a real cluster; only the cluster manager configuration changes.


## 2.7 Summary

- A Spark application has one **Driver** (coordinator) and many **Executors** (workers), managed by a **Cluster Manager**.
- Data is split into **partitions**; each partition is processed by one **task**.
- An action triggers a **Job**, which is split into **Stages** at shuffle boundaries, each Stage made of **Tasks**.
- Transformations are **lazy**; only **actions** trigger real execution, allowing Spark to optimize the whole pipeline first.
- Fault tolerance comes from **lineage** — recomputing lost partitions from recorded transformations.

## Check Your Understanding
1. What triggers a "Job" in Spark?
2. Why are transformations lazy? What advantage does this give the Catalyst optimizer?
3. What is a partition, and how does it relate to parallelism?
4. What cluster manager will we use for this course, and why?

---
**Next:** [03_Installing_PySpark.ipynb](03_Installing_PySpark.ipynb) — Getting a working PySpark environment set up.
